# Deep Dive: Manual Graph Features vs GNN Embeddings

## Research Question
**Why do manually engineered graph statistics (0.6655 AUC-PR) outperform neural graph embeddings (0.57 AUC-PR)?**

This notebook investigates:
1. What graph features were engineered
2. Feature importance: which graph stats matter most?
3. When does graph structure help fraud detection?
4. Why GNN embeddings underperform

**Key Theory from Docs**: Manual graph features extract interpretable, high-signal statistics that outperform 64-dim neural embeddings in sparse graphs.

In [ ]:
import polars as pl
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

## 1. Load Data

In [ ]:
# Load listing nodes with all features
df_listings = pl.read_parquet("../artifacts/nodes_listing.parquet")
print(f"Listings: {len(df_listings):,}")
print(f"Fraud rate: {df_listings['is_fraud'].mean() * 100:.2f}%")

# Load graph features if available
from pathlib import Path
graph_features_path = Path("../artifacts/listing_graph_features.parquet")
if graph_features_path.exists():
    df_graph_features = pl.read_parquet(graph_features_path)
    print(f"\nGraph features loaded: {len(df_graph_features):,} listings")
    
    # Join with listings
    df_full = df_listings.join(df_graph_features, on="object_reference", how="left")
    print(f"Joined dataset: {len(df_full):,} listings")
    
    # List graph features
    graph_cols = [c for c in df_graph_features.columns if c != 'object_reference']
    print(f"\nGraph feature columns ({len(graph_cols)}): {graph_cols}")
else:
    print("\n⚠️  Graph features not found. Run: python src/cli.py graph-features")
    df_full = df_listings
    graph_cols = []